# Fine tune a language aware banking decision model

This notebook reproduces the repository experiment on Google Colab. It loads BANKING77, creates fixed fitting, validation and calibration splits, trains a TF IDF plus logistic regression baseline, evaluates unchanged Laya and fine tunes Laya's typed decision heads.

The design deliberately asks two smaller questions instead of one 77 option question: first choose one of eight banking families, then choose the intent inside that family. The encoder remains frozen. This is supervised head only fine tuning, not LoRA and not the upstream RLCD recipe.

**What gets written:** dataset splits, training records, restart checkpoints, validation predictions, the best complete Laya checkpoint and summary figures are stored in Google Drive. The official BANKING77 test split is loaded for provenance but is not used in training or the reported validation comparison.


## 0. Mount Google Drive and configure the run

Run this before importing Hugging Face libraries so their cache paths point to Drive. `RUN_NAME` identifies one reproducible experiment. Keep it unchanged when resuming. `TARGET_TOTAL_EPOCHS` may be increased later. `MAX_RUN_MINUTES` lets the notebook finish the current epoch and stop cleanly before a Colab session limit.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Banking AI Lab/language-aware-decisions-for-banking")
RUN_NAME = "laya_banking77_head_only_v1"
TARGET_TOTAL_EPOCHS = 30
MAX_RUN_MINUTES = 100

INITIAL_EPOCHS = 2
INITIAL_LEARNING_RATE = 1e-4
CONTINUATION_LEARNING_RATE = 5e-5
MICRO_BATCH = 2
GRAD_ACCUM = 8
SEED = 42

DATA_DIR = PROJECT_ROOT / "data" / "splits"
TRAINING_DIR = PROJECT_ROOT / "data" / "laya_training_v1"
RUN_DIR = PROJECT_ROOT / "runs" / RUN_NAME
HEAD_DIR = RUN_DIR / "epoch_heads"
MODEL_DIR = PROJECT_ROOT / "models" / RUN_NAME
BEST_DIR = MODEL_DIR / "best_laya"
HF_HOME = PROJECT_ROOT / "cache" / "huggingface"
HUB_CACHE = HF_HOME / "hub"
DATASET_CACHE = HF_HOME / "datasets"

for path in (DATA_DIR, TRAINING_DIR, RUN_DIR, HEAD_DIR, MODEL_DIR, HUB_CACHE, DATASET_CACHE):
    path.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HUB_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(DATASET_CACHE)
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

print("Project:", PROJECT_ROOT)
print("Run:", RUN_DIR)


## 1. Install the pinned environment

The experiment was run with Laya 0.3.11. Pinning the fast moving library is important because the notebook uses its typed question preprocessing and model builder. After installation, Colab may ask for a runtime restart only if it replaced a package already imported.


In [ ]:
%env USE_TF=0
%pip -q install "laya==0.3.11" "datasets>=3.0,<5" "transformers>=4.48,<6" "huggingface_hub>=0.30" "safetensors>=0.5" "numpy>=1.26,<3" "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" "joblib>=1.4"


## 2. Import libraries and record the runtime

This cell verifies the GPU and fixes random seeds. A T4 class GPU is sufficient. The classical baseline will run on CPU, while Laya training and evaluation require CUDA for a practical runtime.


In [ ]:
import gc
import hashlib
import json
import random
import shutil
import time

import laya
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import snapshot_download
from joblib import dump
from safetensors.torch import load_file, save_file
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

from laya.agent import _fix_tokenizer_config
from laya.common import QTYPES, build_model, build_sequence, render_options

assert torch.cuda.is_available(), "In Colab select Runtime > Change runtime type > GPU"
assert laya.__version__ == "0.3.11"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0))
print("Laya:", laya.__version__)


## 3. Load BANKING77 and create fixed splits

BANKING77 contains short customer service queries across 77 intents. The MTEB mirror used here currently yields 9,993 training messages, which become 7,994 fitting, 999 validation and 1,000 calibration examples with seed 42. Existing CSVs are reused so a resumed run cannot silently change its data.

Validation selects checkpoints. Calibration is reserved for later confidence work. The official test split remains untouched by this demonstration.


In [ ]:
CATEGORIES = ['card_arrival', 'card_delivery_estimate', 'get_physical_card', 'order_physical_card', 'card_linking', 'getting_spare_card', 'card_about_to_expire', 'activate_my_card', 'card_not_working', 'card_acceptance', 'contactless_not_working', 'pin_blocked', 'change_pin', 'visa_or_mastercard', 'getting_virtual_card', 'virtual_card_not_working', 'get_disposable_virtual_card', 'disposable_card_limits', 'apple_pay_or_google_pay', 'card_payment_wrong_exchange_rate', 'extra_charge_on_statement', 'card_payment_fee_charged', 'card_payment_not_recognised', 'pending_card_payment', 'request_refund', 'Refund_not_showing_up', 'declined_card_payment', 'reverted_card_payment?', 'transaction_charged_twice', 'direct_debit_payment_not_recognised', 'pending_cash_withdrawal', 'wrong_amount_of_cash_received', 'atm_support', 'declined_cash_withdrawal', 'card_swallowed', 'wrong_exchange_rate_for_cash_withdrawal', 'cash_withdrawal_not_recognised', 'cash_withdrawal_charge', 'cancel_transfer', 'transfer_not_received_by_recipient', 'declined_transfer', 'pending_transfer', 'transfer_timing', 'beneficiary_not_allowed', 'transfer_fee_charged', 'receiving_money', 'failed_transfer', 'transfer_into_account', 'balance_not_updated_after_bank_transfer', 'automatic_top_up', 'top_up_by_bank_transfer_charge', 'pending_top_up', 'top_up_limits', 'supported_cards_and_currencies', 'top_up_reverted', 'topping_up_by_card', 'verify_top_up', 'top_up_by_cash_or_cheque', 'top_up_failed', 'top_up_by_card_charge', 'balance_not_updated_after_cheque_or_cash_deposit', 'exchange_rate', 'fiat_currency_support', 'exchange_via_app', 'exchange_charge', 'age_limit', 'edit_personal_details', 'why_verify_identity', 'unable_to_verify_identity', 'terminate_account', 'verify_source_of_funds', 'passcode_forgotten', 'verify_my_identity', 'country_support', 'lost_or_stolen_card', 'compromised_card', 'lost_or_stolen_phone']
assert len(CATEGORIES) == len(set(CATEGORIES)) == 77
LABEL2ID = {label: index for index, label in enumerate(CATEGORIES)}
NORMALIZED_TO_CANONICAL = {label.lower().replace("?", ""): label for label in CATEGORIES}

def to_frame(split):
    frame = split.to_pandas().copy()
    text_column = "text" if "text" in frame.columns else "sentence"
    frame = frame.rename(columns={text_column: "text"})
    if "label_text" in frame.columns:
        normalized = frame["label_text"].astype(str).str.lower().str.replace("?", "", regex=False)
        frame["label_name"] = normalized.map(NORMALIZED_TO_CANONICAL)
    elif not np.issubdtype(frame["label"].dtype, np.number):
        frame["label_name"] = frame["label"].astype(str)
    else:
        frame["label_name"] = frame["label"].astype(int).map(dict(enumerate(CATEGORIES)))
    frame["label"] = frame["label_name"].map(LABEL2ID)
    assert frame[["text", "label", "label_name"]].notna().all().all()
    return frame[["text", "label", "label_name"]].reset_index(drop=True)

dataset = load_dataset("mteb/banking77", cache_dir=str(DATASET_CACHE))
train_all = to_frame(dataset["train"])
official_test = to_frame(dataset["test"])

split_paths = {name: DATA_DIR / f"{name}.csv" for name in ("fit", "validation", "calibration")}
if all(path.is_file() for path in split_paths.values()):
    fit = pd.read_csv(split_paths["fit"])
    validation = pd.read_csv(split_paths["validation"])
    calibration = pd.read_csv(split_paths["calibration"])
    print("Reused existing split files")
else:
    all_indices = np.arange(len(train_all))
    fit_indices, remainder_indices = train_test_split(
        all_indices, test_size=0.20, random_state=SEED, stratify=train_all["label"]
    )
    validation_indices, calibration_indices = train_test_split(
        remainder_indices,
        test_size=0.50,
        random_state=SEED,
        stratify=train_all.iloc[remainder_indices]["label"],
    )
    fit = train_all.iloc[fit_indices].reset_index(drop=True)
    validation = train_all.iloc[validation_indices].reset_index(drop=True)
    calibration = train_all.iloc[calibration_indices].reset_index(drop=True)
    for name, frame in {"fit": fit, "validation": validation, "calibration": calibration}.items():
        frame.to_csv(split_paths[name], index=False)

for name, frame in {"fit": fit, "validation": validation, "calibration": calibration, "official_test": official_test}.items():
    assert frame["label_name"].nunique() == 77
    print(f"{name:>12}: {len(frame):,} messages")


## 4. Define the two stage banking decision hierarchy

Laya allocates a fixed option budget to each typed choice. A flat 77 option prompt leaves little space for each label, so this experiment first predicts one of eight service families and then the intent within that family. Every fitting message becomes exactly two supervised typed decisions.

The resulting `questions.json` is part of the model interface. The same wording and option order must be used during training and inference.


In [ ]:
FAMILIES = {'card_lifecycle_access': ['card_arrival', 'card_delivery_estimate', 'get_physical_card', 'order_physical_card', 'card_linking', 'getting_spare_card', 'card_about_to_expire', 'activate_my_card', 'card_not_working', 'card_acceptance', 'contactless_not_working', 'pin_blocked', 'change_pin', 'visa_or_mastercard', 'getting_virtual_card', 'virtual_card_not_working', 'get_disposable_virtual_card', 'disposable_card_limits', 'apple_pay_or_google_pay'], 'payments_refunds_direct_debits': ['card_payment_wrong_exchange_rate', 'extra_charge_on_statement', 'card_payment_fee_charged', 'card_payment_not_recognised', 'pending_card_payment', 'request_refund', 'Refund_not_showing_up', 'declined_card_payment', 'reverted_card_payment?', 'transaction_charged_twice', 'direct_debit_payment_not_recognised'], 'cash_withdrawals_atms': ['pending_cash_withdrawal', 'wrong_amount_of_cash_received', 'atm_support', 'declined_cash_withdrawal', 'card_swallowed', 'wrong_exchange_rate_for_cash_withdrawal', 'cash_withdrawal_not_recognised', 'cash_withdrawal_charge'], 'bank_transfers_beneficiaries': ['cancel_transfer', 'transfer_not_received_by_recipient', 'declined_transfer', 'pending_transfer', 'transfer_timing', 'beneficiary_not_allowed', 'transfer_fee_charged', 'receiving_money', 'failed_transfer', 'transfer_into_account', 'balance_not_updated_after_bank_transfer'], 'topups_deposits': ['automatic_top_up', 'top_up_by_bank_transfer_charge', 'pending_top_up', 'top_up_limits', 'supported_cards_and_currencies', 'top_up_reverted', 'topping_up_by_card', 'verify_top_up', 'top_up_by_cash_or_cheque', 'top_up_failed', 'top_up_by_card_charge', 'balance_not_updated_after_cheque_or_cash_deposit'], 'currency_exchange': ['exchange_rate', 'fiat_currency_support', 'exchange_via_app', 'exchange_charge'], 'identity_account_eligibility': ['age_limit', 'edit_personal_details', 'why_verify_identity', 'unable_to_verify_identity', 'terminate_account', 'verify_source_of_funds', 'passcode_forgotten', 'verify_my_identity', 'country_support'], 'security_lost_devices': ['lost_or_stolen_card', 'compromised_card', 'lost_or_stolen_phone']}
FAMILY_DESCRIPTIONS = {'card_lifecycle_access': 'obtaining, receiving, activating, linking or using physical and virtual cards', 'payments_refunds_direct_debits': 'card payments, duplicate charges, refunds and direct debit transactions', 'cash_withdrawals_atms': 'cash machines, ATM withdrawals, fees, failures or retained cards', 'bank_transfers_beneficiaries': 'bank transfers, recipients, beneficiaries, timing, fees and failures', 'topups_deposits': 'adding money by card, bank transfer, cash or cheque and related limits or failures', 'currency_exchange': 'supported currencies, exchange rates, exchange fees and exchanging in the app', 'identity_account_eligibility': 'identity verification, account profile, access, eligibility and country support', 'security_lost_devices': 'lost or stolen cards and phones, or compromised card security'}
INTENT_TO_FAMILY = {intent: family for family, intents in FAMILIES.items() for intent in intents}
assert len(FAMILIES) == 8 and len(INTENT_TO_FAMILY) == 77

def readable(label):
    return label.replace("_", " ").replace("?", "").strip()

QUESTIONS = {
    "family": {
        "type": "choice",
        "instructions": "Which banking service family best matches the customer's request?",
        "criteria": {family: FAMILY_DESCRIPTIONS[family] for family in FAMILIES},
    }
}
for family, intents in FAMILIES.items():
    QUESTIONS[f"intent__{family}"] = {
        "type": "choice",
        "instructions": f"If this request belongs to {readable(family)}, which specific intent fits best?",
        "criteria": {intent: readable(intent) for intent in intents},
    }

QUESTIONS_PATH = TRAINING_DIR / "questions.json"
QUESTIONS_PATH.write_text(json.dumps(QUESTIONS, indent=2), encoding="utf-8")

def one_hot(keys, correct):
    return {key: float(key == correct) for key in keys}

def training_record(text, intent):
    family = INTENT_TO_FAMILY[intent]
    family_question = QUESTIONS["family"]
    intent_question = QUESTIONS[f"intent__{family}"]
    gold = {
        "family": {"probabilities": one_hot(family_question["criteria"], family)},
        f"intent__{family}": {"probabilities": one_hot(intent_question["criteria"], intent)},
    }
    return {
        "state": json.dumps({"message": text}, ensure_ascii=False),
        "questions": json.dumps({"family": family_question, f"intent__{family}": intent_question}, ensure_ascii=False),
        "gold": json.dumps(gold, ensure_ascii=False),
    }

FIT_JSONL = TRAINING_DIR / "fit.jsonl"
with FIT_JSONL.open("w", encoding="utf-8") as stream:
    for row in fit.itertuples(index=False):
        stream.write(json.dumps(training_record(row.text, row.label_name), ensure_ascii=False) + "\n")
print("Saved", len(fit), "messages and", 2 * len(fit), "typed decisions")


## 5. Train the conventional machine learning baseline

TF IDF converts words and word pairs into sparse numerical features. Logistic regression learns which features support each of the 77 labels. It does not have a pretrained language representation, but it is an excellent fit for a stable in domain taxonomy and provides the sanity check this experiment needs.

The fitted vectorizer and classifier are saved together. Expect training to take seconds rather than hours.


In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2), min_df=2, max_features=100_000,
    sublinear_tf=True, strip_accents="unicode",
)
classifier = LogisticRegression(C=4.0, max_iter=2_000, solver="lbfgs")

baseline_started = time.perf_counter()
X_fit = tfidf.fit_transform(fit["text"])
classifier.fit(X_fit, fit["label"])
baseline_training_seconds = time.perf_counter() - baseline_started

baseline_started = time.perf_counter()
baseline_prediction = classifier.predict(tfidf.transform(validation["text"]))
baseline_inference_seconds = time.perf_counter() - baseline_started
baseline_metrics = {
    "model": "TF IDF + logistic regression",
    "accuracy": float(accuracy_score(validation["label"], baseline_prediction)),
    "macro_f1": float(f1_score(validation["label"], baseline_prediction, average="macro")),
    "training_seconds": baseline_training_seconds,
    "inference_seconds": baseline_inference_seconds,
}
dump({"vectorizer": tfidf, "classifier": classifier}, MODEL_DIR / "tfidf_logreg.joblib")
print(json.dumps(baseline_metrics, indent=2))


## 6. Evaluate unchanged Laya on exactly the same validation messages

Laya answers the family question and all family specific intent questions in one typed prediction call. The probabilities are composed as `P(family) × P(intent | family)` and normalized across all 77 intents.

This before score is the honest starting point. It is saved before any banking weight update.


In [ ]:
MODEL_ID = "convaiinnovations/laya"
families = list(QUESTIONS["family"]["criteria"])
labels = sorted(intent for family in families for intent in QUESTIONS[f"intent__{family}"]["criteria"])

def compose(result):
    answers = result["answers"]
    family_probabilities = answers["family"]["probabilities"]
    probabilities = {
        intent: float(family_probabilities[family])
        * float(answers[f"intent__{family}"]["probabilities"][intent])
        for family in families
        for intent in QUESTIONS[f"intent__{family}"]["criteria"]
    }
    vector = np.asarray([probabilities[label] for label in labels], dtype=float)
    return vector / vector.sum()

def evaluate_agent(agent, frame):
    torch.cuda.synchronize()
    started = time.perf_counter()
    results = agent.predict_batch([{"message": text} for text in frame["text"]], QUESTIONS, batch_size=4)
    torch.cuda.synchronize()
    seconds = time.perf_counter() - started
    probabilities = np.vstack([compose(result) for result in results])
    predicted = [labels[index] for index in probabilities.argmax(axis=1)]
    metrics = {
        "accuracy": float(accuracy_score(frame["label_name"], predicted)),
        "macro_f1": float(f1_score(frame["label_name"], predicted, labels=labels, average="macro")),
        "inference_seconds": seconds,
    }
    return predicted, probabilities, metrics

unchanged_agent = laya.load(MODEL_ID, device="cuda")
unchanged_prediction, unchanged_probabilities, unchanged_metrics = evaluate_agent(unchanged_agent, validation)
pd.DataFrame({
    "text": validation["text"], "expected": validation["label_name"],
    "predicted": unchanged_prediction,
    "score_uncalibrated": unchanged_probabilities.max(axis=1),
}).to_csv(RUN_DIR / "validation_laya_unchanged.csv", index=False)
(RUN_DIR / "laya_unchanged_metrics.json").write_text(json.dumps(unchanged_metrics, indent=2))
print(json.dumps(unchanged_metrics, indent=2))
del unchanged_agent, unchanged_probabilities
gc.collect()
torch.cuda.empty_cache()


## 7. Tokenize typed decisions and restore any completed training

The base checkpoint is downloaded once and cached on Drive. Laya's own sequence builder renders each question and marks the option positions used by the decision head.

On a new run, training starts from the public checkpoint. On a resumed run, the cell restores the head, optimizer, gradient scaler and random states from the latest completed epoch. Changing data or optimization settings requires a new `RUN_NAME`.


In [ ]:
snapshot = Path(snapshot_download(MODEL_ID, cache_dir=str(HUB_CACHE)))
_fix_tokenizer_config(str(snapshot))
tokenizer = AutoTokenizer.from_pretrained(str(snapshot / "tokenizer"))
cfg = json.loads((snapshot / "rl_agent_config.json").read_text())

with FIT_JSONL.open(encoding="utf-8") as stream:
    records = [json.loads(line) for line in stream]

def build_item(state, question, gold):
    criteria = question["criteria"]
    keys = list(criteria)
    target = [float(gold["probabilities"][key]) for key in keys]
    sequence, markers = build_sequence(
        tokenizer, state,
        {"t": "choice", "ins": question["instructions"], "crit": criteria},
        cfg["max_len"], cfg["head_max_len"],
    )
    if len(markers) != len(keys) or len(markers) != len(render_options({"t": "choice", "crit": criteria})):
        raise ValueError("A choice option was truncated; inspect head_max_len")
    return {"ids": sequence, "markers": markers, "target": target}

items = []
for record in records:
    state, questions, gold = [json.loads(record[key]) for key in ("state", "questions", "gold")]
    for question_id, question in questions.items():
        items.append(build_item(state, question, gold[question_id]))
assert len(items) == 2 * len(fit)

def collate(chunk):
    length = max(len(item["ids"]) for item in chunk)
    option_count = max(len(item["markers"]) for item in chunk)
    ids = torch.full((len(chunk), length), tokenizer.pad_token_id, dtype=torch.long)
    attention = torch.zeros_like(ids)
    markers = torch.zeros((len(chunk), option_count), dtype=torch.long)
    mask = torch.zeros((len(chunk), option_count), dtype=torch.bool)
    target = torch.zeros((len(chunk), option_count), dtype=torch.float32)
    for index, item in enumerate(chunk):
        ids[index, :len(item["ids"])] = torch.tensor(item["ids"])
        attention[index, :len(item["ids"])] = 1
        count = len(item["markers"])
        markers[index, :count] = torch.tensor(item["markers"])
        mask[index, :count] = True
        target[index, :count] = torch.tensor(item["target"])
    return ids, attention, markers, mask, target

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

fingerprints = {
    "fit_jsonl": file_sha256(FIT_JSONL),
    "questions": file_sha256(QUESTIONS_PATH),
    "validation": file_sha256(DATA_DIR / "validation.csv"),
}
settings = {
    "model_id": MODEL_ID,
    "initial_epochs": INITIAL_EPOCHS,
    "initial_learning_rate": INITIAL_LEARNING_RATE,
    "continuation_learning_rate": CONTINUATION_LEARNING_RATE,
    "micro_batch": MICRO_BATCH,
    "gradient_accumulation": GRAD_ACCUM,
    "fingerprints": fingerprints,
    "training_mode": "frozen encoder, supervised cross entropy on typed decision heads",
}
settings_path = RUN_DIR / "settings.json"
if settings_path.exists():
    assert json.loads(settings_path.read_text()) == settings, "Inputs changed; choose a new RUN_NAME"
else:
    settings_path.write_text(json.dumps(settings, indent=2))

model = build_model(cfg, encoder_dir=str(snapshot / "encoder"))
model.load_state_dict(load_file(str(snapshot / "model.safetensors")), strict=True)
for parameter in model.encoder.parameters():
    parameter.requires_grad_(False)
HEAD_KEYS = [key for key in model.state_dict() if not key.startswith("encoder.")]
model.to("cuda")
model.train()
model.encoder.eval()
trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
print("Trainable parameters:", sum(parameter.numel() for parameter in trainable))

RESUME_PATH = RUN_DIR / "resume_state.pt"
HISTORY_PATH = RUN_DIR / "epoch_history.csv"
completed_epoch = 0
current_learning_rate = INITIAL_LEARNING_RATE
optimizer = torch.optim.AdamW(trainable, lr=current_learning_rate, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda")

if RESUME_PATH.exists():
    resume = torch.load(RESUME_PATH, map_location="cpu", weights_only=False)
    assert resume["fingerprints"] == fingerprints
    completed_epoch = int(resume["epoch"])
    current_learning_rate = CONTINUATION_LEARNING_RATE if completed_epoch >= INITIAL_EPOCHS else INITIAL_LEARNING_RATE
    optimizer = torch.optim.AdamW(trainable, lr=current_learning_rate, weight_decay=0.01)
    missing, unexpected = model.load_state_dict(resume["head_state"], strict=False)
    assert not unexpected and set(missing) == set(model.state_dict()) - set(HEAD_KEYS)
    optimizer.load_state_dict(resume["optimizer"])
    scaler.load_state_dict(resume["scaler"])
    torch.set_rng_state(resume["torch_rng"])
    torch.cuda.set_rng_state_all(resume["cuda_rng"])
    random.setstate(resume["python_rng"])
    print("Resumed after epoch", completed_epoch)
else:
    history = [{
        "epoch": 0, "mean_training_loss": np.nan,
        "accuracy": unchanged_metrics["accuracy"], "macro_f1": unchanged_metrics["macro_f1"],
        "training_seconds": 0.0, "inference_seconds": unchanged_metrics["inference_seconds"],
        "head_checkpoint": MODEL_ID,
    }]
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    print("Starting from the unchanged public checkpoint")


## 8. Fine tune, validate and checkpoint at each epoch

The first two epochs use the original pilot learning rate. At epoch three the optimizer is intentionally restarted with a smaller learning rate, matching the experiment that produced the published checkpoint.

Each completed epoch first saves the head and optimizer state, then evaluates all 999 validation messages. If Colab stops during validation, rerunning the notebook completes that validation before training the next epoch. The best full checkpoint is selected by validation accuracy, with macro F1 as a tie breaker.


In [ ]:
LOCAL_EVAL_DIR = Path("/content/laya_epoch_evaluation")
LOCAL_EVAL_DIR.mkdir(parents=True, exist_ok=True)
for folder in ("encoder", "tokenizer"):
    shutil.copytree(snapshot / folder, LOCAL_EVAL_DIR / folder, dirs_exist_ok=True)
shutil.copy2(snapshot / "rl_agent_config.json", LOCAL_EVAL_DIR / "rl_agent_config.json")

def export_local_model():
    model.to("cpu")
    gc.collect()
    torch.cuda.empty_cache()
    state = {key: value.detach().half().contiguous().cpu() for key, value in model.state_dict().items()}
    save_file(state, str(LOCAL_EVAL_DIR / "model.safetensors"))
    del state

def validate_epoch(epoch):
    export_local_model()
    agent = laya.load(str(LOCAL_EVAL_DIR), device="cuda")
    print(f"Epoch {epoch}: evaluating {len(validation)} held out messages")
    predicted, probabilities, metrics = evaluate_agent(agent, validation)
    pd.DataFrame({
        "text": validation["text"], "expected": validation["label_name"],
        "predicted": predicted, "score_uncalibrated": probabilities.max(axis=1),
    }).to_csv(RUN_DIR / f"validation_epoch_{epoch:03d}.csv", index=False)
    del agent, probabilities
    gc.collect()
    torch.cuda.empty_cache()
    model.to("cuda")
    model.train()
    model.encoder.eval()
    return metrics

def load_history():
    return pd.read_csv(HISTORY_PATH).to_dict("records")

def write_history(rows):
    temporary = RUN_DIR / "epoch_history.csv.partial"
    pd.DataFrame(rows).to_csv(temporary, index=False)
    os.replace(temporary, HISTORY_PATH)

def current_winner(rows):
    return max(rows, key=lambda row: (float(row["accuracy"]), float(row["macro_f1"])))

def save_training_state(epoch):
    head = {key: model.state_dict()[key].detach().contiguous().cpu() for key in HEAD_KEYS}
    save_file({key: value.half() for key, value in head.items()}, str(HEAD_DIR / f"head_epoch_{epoch:03d}.safetensors"))
    payload = {
        "epoch": epoch, "head_state": head, "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(), "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all(), "python_rng": random.getstate(),
        "fingerprints": fingerprints,
    }
    temporary = Path("/content/laya_resume_state.pt")
    torch.save(payload, temporary)
    shutil.copy2(temporary, RUN_DIR / "resume_state.pt.partial")
    os.replace(RUN_DIR / "resume_state.pt.partial", RESUME_PATH)

def save_best(epoch, metrics):
    BEST_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(LOCAL_EVAL_DIR / "model.safetensors", BEST_DIR / "model.safetensors")
    for folder in ("encoder", "tokenizer"):
        shutil.copytree(LOCAL_EVAL_DIR / folder, BEST_DIR / folder, dirs_exist_ok=True)
    shutil.copy2(LOCAL_EVAL_DIR / "rl_agent_config.json", BEST_DIR / "rl_agent_config.json")
    (BEST_DIR / "selected_epoch.json").write_text(json.dumps({
        "epoch": epoch,
        "validation_accuracy": metrics["accuracy"],
        "validation_macro_f1": metrics["macro_f1"],
    }, indent=2))

def evaluate_pending_epoch():
    rows = load_history()
    if int(rows[-1]["epoch"]) == completed_epoch:
        return
    info = json.loads((RUN_DIR / f"training_epoch_{completed_epoch:03d}.json").read_text())
    metrics = validate_epoch(completed_epoch)
    row = {**info, **metrics, "head_checkpoint": str(HEAD_DIR / f"head_epoch_{completed_epoch:03d}.safetensors")}
    previous = current_winner(rows)
    if (row["accuracy"], row["macro_f1"]) > (previous["accuracy"], previous["macro_f1"]):
        save_best(completed_epoch, metrics)
    rows.append(row)
    write_history(rows)
    print(f"Epoch {completed_epoch}: accuracy {row['accuracy']:.4f}, macro F1 {row['macro_f1']:.4f}; best epoch {int(current_winner(rows)['epoch'])}")

run_started = time.perf_counter()
if completed_epoch > 0:
    evaluate_pending_epoch()

for epoch in range(completed_epoch + 1, TARGET_TOTAL_EPOCHS + 1):
    if epoch == INITIAL_EPOCHS + 1:
        optimizer = torch.optim.AdamW(trainable, lr=CONTINUATION_LEARNING_RATE, weight_decay=0.01)
        scaler = torch.amp.GradScaler("cuda")
        print("Restarted AdamW at the continuation learning rate")

    order = list(range(len(items)))
    random.Random(SEED + epoch - 1).shuffle(order)
    optimizer.zero_grad(set_to_none=True)
    losses = []
    started_epoch = time.perf_counter()
    batch_count = (len(order) + MICRO_BATCH - 1) // MICRO_BATCH

    for batch_index, offset in enumerate(range(0, len(order), MICRO_BATCH)):
        batch = collate([items[index] for index in order[offset:offset + MICRO_BATCH]])
        ids, attention, markers, mask, target = [tensor.to("cuda") for tensor in batch]
        question_types = torch.full((len(ids),), QTYPES["choice"], dtype=torch.long, device="cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            logits, _ = model(ids, attention, markers, mask, question_types)
            log_probabilities = torch.log_softmax(logits.float().masked_fill(~mask, -1e4), dim=-1)
            loss = -(target * log_probabilities).sum(dim=-1).mean()
        scaler.scale(loss / GRAD_ACCUM).backward()
        losses.append(float(loss.detach().cpu()))
        if (batch_index + 1) % GRAD_ACCUM == 0 or offset + MICRO_BATCH >= len(order):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        if (batch_index + 1) % 1000 == 0 or batch_index + 1 == batch_count:
            print(f"Epoch {epoch}, batch {batch_index + 1}/{batch_count}, mean loss {np.mean(losses):.4f}", flush=True)

    torch.cuda.synchronize()
    info = {
        "epoch": epoch,
        "mean_training_loss": float(np.mean(losses)),
        "training_seconds": time.perf_counter() - started_epoch,
    }
    (RUN_DIR / f"training_epoch_{epoch:03d}.json").write_text(json.dumps(info, indent=2))
    save_training_state(epoch)
    completed_epoch = epoch
    evaluate_pending_epoch()

    saved_rng = torch.load(RESUME_PATH, map_location="cpu", weights_only=False)
    torch.set_rng_state(saved_rng["torch_rng"])
    torch.cuda.set_rng_state_all(saved_rng["cuda_rng"])
    random.setstate(saved_rng["python_rng"])
    del saved_rng

    if (time.perf_counter() - run_started) / 60 >= MAX_RUN_MINUTES:
        print("Time budget reached after a completed epoch. Rerun all cells to continue safely.")
        break


## 9. Compare the three approaches and create the result figures

This cell reads the complete epoch history, identifies the best checkpoint and writes a compact comparison table. Macro F1 is the main visual metric because it gives every one of the 77 intents equal weight.

If the run stopped before `TARGET_TOTAL_EPOCHS`, the figures describe the best completed epoch so far. The original public result reached 73.5 percent accuracy and 74.0 percent macro F1 at epoch 30.


In [ ]:
history = pd.read_csv(HISTORY_PATH).sort_values("epoch")
best = history.sort_values(["accuracy", "macro_f1", "epoch"], ascending=[False, False, True]).iloc[0]
best_epoch = int(best["epoch"])

comparison = pd.DataFrame([
    {
        "model": "Laya, zero-shot", "training_examples": 0,
        "validation_examples": len(validation), "accuracy": unchanged_metrics["accuracy"],
        "macro_f1": unchanged_metrics["macro_f1"],
    },
    {
        "model": f"Laya, banking fine tuned, epoch {best_epoch}", "training_examples": len(fit),
        "validation_examples": len(validation), "accuracy": float(best["accuracy"]),
        "macro_f1": float(best["macro_f1"]),
    },
    {
        "model": "TF IDF + logistic regression", "training_examples": len(fit),
        "validation_examples": len(validation), "accuracy": baseline_metrics["accuracy"],
        "macro_f1": baseline_metrics["macro_f1"],
    },
])
comparison.to_csv(RUN_DIR / "model_comparison.csv", index=False)
display(comparison.style.format({"accuracy": "{:.3f}", "macro_f1": "{:.3f}"}))

from matplotlib.ticker import FormatStrFormatter

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 12,
    "axes.titlesize": 16, "axes.labelsize": 12,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "savefig.facecolor": "white",
})

# Match the blog chart: show the fine-tuning phase from epoch 2 onward.
tuned = history[history["epoch"] >= 2]
figure, axis = plt.subplots(figsize=(7.5, 4.0))
axis.plot(tuned["epoch"], tuned["macro_f1"], color="#1f77b4", marker="o", markersize=4)
axis.set_title("Fine-tuning Laya for Banking Decisions")
axis.set_xlabel("Epoch")
axis.set_ylabel("F1 score")
axis.set_xlim(2, 30)
axis.set_ylim(0.55, 0.76)
axis.set_xticks([2, 5, 10, 15, 20, 25, 30])
axis.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
axis.grid(axis="y", linewidth=0.8, alpha=0.7)
axis.set_axisbelow(True)
figure.tight_layout()
figure.savefig(RUN_DIR / "laya_banking_fine_tuning_progress.png", dpi=180, bbox_inches="tight")
plt.show()

labels = ["Laya,\nzero-shot", "Laya, banking\nfine-tuned", "TF-IDF + logistic\nregression"]
figure, axis = plt.subplots(figsize=(7.5, 4.0))
axis.bar(labels, comparison["macro_f1"], width=0.55, color=["#1f77b4", "#ff7f0e", "#2ca02c"])
axis.set_title("Model comparison")
axis.set_ylabel("F1 score")
axis.set_ylim(0, 1)
axis.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
axis.grid(axis="y", linewidth=0.8, alpha=0.7)
axis.set_axisbelow(True)
figure.tight_layout()
figure.savefig(RUN_DIR / "banking_model_macro_f1_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

print("Best epoch:", best_epoch)
print("Best checkpoint:", BEST_DIR if best_epoch > 0 else MODEL_ID)
print("Run outputs:", RUN_DIR)


## 10. Interpretation and next steps

The classical baseline is expected to remain stronger on this narrow, stable taxonomy. The useful signal is that a general typed decision model can be adapted substantially with one public dataset, a frozen encoder and a single Colab GPU.

Before operational use, evaluate the frozen design once on the official test set, calibrate confidence on the reserved calibration split, add out of distribution messages, measure latency and throughput, and define a human review policy. The stored Laya temperatures currently trigger a warning and should not be treated as production calibrated confidence.
